# E-Commerce Return Prediction — Data Cleaning & Validation

## Objective

This notebook prepares the raw e-commerce order dataset for downstream
SQL analysis, exploratory data analysis, Power BI visualization, and
machine learning.

The primary goals of this stage are to:

- Understand the structure and quality of the raw dataset
- Identify missing, duplicate, invalid, and inconsistent records
- Apply appropriate data-cleaning rules
- Validate the cleaned dataset
- Export a reliable dataset for downstream analysis

### Data Preparation Pipeline

**Raw Dataset → Data Profiling → Cleaning → Validation → Clean Dataset**

## 1. Business Context

E-commerce businesses generate large volumes of transactional data, but
raw transaction data may contain missing values, invalid numerical values,
duplicate records, inconsistent categories, and other data-quality issues.

Since this project aims to analyze and predict product returns, the quality
of the input data is particularly important. Incorrect or inconsistent
records could lead to misleading return-rate calculations and unreliable
machine-learning predictions.

Therefore, data cleaning is performed **before the dataset is loaded into
the SQL database**.

### Key Principle

> **The SQL database should contain validated analytical data rather than
> raw, unverified records.**

## Import Required Libraries

The following libraries are used for data loading, inspection, cleaning,
validation, and exporting the processed dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_PATH = Path("../data/raw/ecommerce_messy_data.csv")
CLEANED_PATH = Path("../data/cleaned/ecommerce_cleaned.csv")
OUTPUT_PATH = Path("../outputs/cleaning_validation_summary.csv")

## Load the Raw Dataset

The original dataset is loaded without modifying the source data.

A separate cleaned copy will be created during the preprocessing stage so
that the original dataset remains available for comparison and auditing.

### Input

`data/raw/ecommerce_raw.csv`

### Output

A Pandas DataFrame containing the original transaction records.

In [3]:
df = pd.read_csv(RAW_PATH)

df.head()

,OrderID,CustomerID,OrderDate,ProductID,ProductCategory,ProductName,Quantity,PricePerUnit,PaymentMethod,OrderStatus,CustomerLocation,CustomerSegment,DiscountApplied,DeliveryTime(days),IsReturned,TotalAmount
0,ORD100000,CUST1102,2023-06-26,PROD2284,Electrnics,Product_86,4.0,3968.22,Credit Card,Delivered,Bangalore,New,0.18,4,0,13015.76
1,ORD100001,CUST1435,2023-09-24,PROD2156,Books,Product_92,3.0,1554.25,Net Banking,Returned,Chennai,New,0.20,5,1,3730.20
2,ORD100002,CUST1348,2023-12-07,PROD2074,Books,Product_14,4.0,2752.71,Cash on Delivery,Returned,Delhi,VIP,0.05,-5,1,10460.30
3,ORD100003,CUST1270,2023-10-09,PROD2287,Books,Product_27,1.0,NaN,Credit Card,Delivered,Chennai,New,0.29,1,0,1272.06
4,ORD100004,CUST1106,2023-03-04,PROD2129,Books,Product_40,NaN,NaN,Net Banking,NaN,Chennai,Regular,0.16,7,0,5366.54


### Data Types

Correct data types are important for both analysis and downstream database
storage.

For example:

- Dates should be represented as date/datetime values.
- Quantities should be numeric.
- Monetary values should be numeric.
- Categorical variables should contain consistent text values.
- The return indicator should be represented as a binary variable.

In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 1020
Columns: 16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   OrderID             1020 non-null   object 
 1   CustomerID          1020 non-null   object 
 2   OrderDate           1020 non-null   object 
 3   ProductID           1020 non-null   object 
 4   ProductCategory     1020 non-null   object 
 5   ProductName         1020 non-null   object 
 6   Quantity            918 non-null    float64
 7   PricePerUnit        919 non-null    float64
 8   PaymentMethod       1020 non-null   object 
 9   OrderStatus         970 non-null    object 
 10  CustomerLocation    1020 non-null   object 
 11  CustomerSegment     1020 non-null   object 
 12  DiscountApplied     1020 non-null   float64
 13  DeliveryTime(days)  1020 non-null   int64  
 14  IsReturned          1020 non-null   int64  
 15  TotalAmount         1020 non-nul

In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OrderID,1020,1000,ORD100136,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CustomerID,1020,423,CUST1098,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OrderDate,1020,345,2023-04-21,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductID,1020,287,PROD2083,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductCategory,1020,6,Electrnics,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ProductName,1020,99,Product_47,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,918.0,NaN,NaN,NaN,2.465142,1.127811,1.0,1.0,2.0,3.0,4.0
PricePerUnit,919.0,NaN,NaN,NaN,2558.656921,1402.092481,106.6,1361.685,2610.2,3773.665,4988.35
PaymentMethod,1020,4,Net Banking,273,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OrderStatus,970,3,Returned,345,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Missing Value Analysis

Missing values are evaluated column by column before deciding how they
should be handled.

Missing values are not automatically replaced. The appropriate treatment
depends on:

- The meaning of the variable
- The proportion of missing observations
- Whether the variable is required for analysis
- Whether a meaningful replacement value exists

In [6]:
missing_summary = (
    df.isnull()
      .sum()
      .to_frame("Missing_Count")
)

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Count"] / len(df) * 100
)

missing_summary.sort_values(
    "Missing_Count",
    ascending=False
)

,Missing_Count,Missing_Percentage
Quantity,102,10.000000
PricePerUnit,101,9.901961
OrderStatus,50,4.901961
OrderID,0,0.000000
ProductID,0,0.000000
OrderDate,0,0.000000
ProductName,0,0.000000
CustomerID,0,0.000000
ProductCategory,0,0.000000
PaymentMethod,0,0.000000


## Duplicate Record Analysis

Duplicate records can artificially inflate order counts, revenue,
and return rates.

Therefore, duplicate records are identified before further analysis.

The following checks determine:

- Whether completely duplicated rows exist
- The number of duplicate records
- Whether the order identifier contains duplicates

In [7]:
duplicate_rows = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_rows)

Exact duplicate rows: 20


In [8]:
duplicate_order_ids = df["OrderID"].duplicated().sum()

print("Duplicate OrderIDs:", duplicate_order_ids)

Duplicate OrderIDs: 20


In [9]:
duplicate_orders = df[
    df["OrderID"].duplicated(keep=False)
].sort_values("OrderID")

duplicate_orders[
    [
        "OrderID",
        "Quantity",
        "PricePerUnit",
        "ProductCategory",
        "IsReturned",
        "OrderStatus"
    ]
]

,OrderID,Quantity,PricePerUnit,ProductCategory,IsReturned,OrderStatus
76,ORD100076,3.0,2773.37,Home & Kitchen,1,Returned
1011,ORD100076,3.0,2773.37,Home & Kitchen,1,Returned
136,ORD100136,4.0,2964.87,Fasion,1,Returned
1009,ORD100136,4.0,2964.87,Fasion,1,Returned
280,ORD100280,2.0,4020.36,Beauty,0,Delivered
1016,ORD100280,2.0,4020.36,Beauty,0,Delivered
1019,ORD100319,1.0,1582.15,Toys,1,Returned
319,ORD100319,1.0,1582.15,Toys,1,Returned
411,ORD100411,3.0,1237.07,Beauty,1,Returned
1004,ORD100411,3.0,1237.07,Beauty,1,Returned


In [10]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace("(", "", regex=False)
      .str.replace(")", "", regex=False)
      .str.replace("/", "_", regex=False)
      .str.replace(" ", "_", regex=False)
)

df.columns.tolist()

['orderid',
 'customerid',
 'orderdate',
 'productid',
 'productcategory',
 'productname',
 'quantity',
 'priceperunit',
 'paymentmethod',
 'orderstatus',
 'customerlocation',
 'customersegment',
 'discountapplied',
 'deliverytimedays',
 'isreturned',
 'totalamount']

In [11]:
columns_to_drop = [
    "customerid",
    "productid",
    "productname",
    "orderstatus"
]

df = df.drop(columns=columns_to_drop)

print(df.shape)
df.head()

(1020, 12)


,orderid,orderdate,productcategory,quantity,priceperunit,paymentmethod,customerlocation,customersegment,discountapplied,deliverytimedays,isreturned,totalamount
0,ORD100000,2023-06-26,Electrnics,4.0,3968.22,Credit Card,Bangalore,New,0.18,4,0,13015.76
1,ORD100001,2023-09-24,Books,3.0,1554.25,Net Banking,Chennai,New,0.20,5,1,3730.20
2,ORD100002,2023-12-07,Books,4.0,2752.71,Cash on Delivery,Delhi,VIP,0.05,-5,1,10460.30
3,ORD100003,2023-10-09,Books,1.0,NaN,Credit Card,Chennai,New,0.29,1,0,1272.06
4,ORD100004,2023-03-04,Books,NaN,NaN,Net Banking,Chennai,Regular,0.16,7,0,5366.54


In [12]:
before_duplicates = len(df)

df = df.drop_duplicates(
    subset="orderid",
    keep="first"
)

after_duplicates = len(df)

print("Rows before removing duplicates:", before_duplicates)
print("Rows after removing duplicates:", after_duplicates)
print("Duplicates removed:", before_duplicates - after_duplicates)

Rows before removing duplicates: 1020
Rows after removing duplicates: 1000
Duplicates removed: 20


In [13]:
df["orderdate"] = pd.to_datetime(
    df["orderdate"],
    errors="coerce"
)

In [14]:
print("Invalid dates:", df["orderdate"].isna().sum())

Invalid dates: 0


In [15]:
numeric_columns = df.select_dtypes(include=np.number)

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

## Categorical Data Validation

Categorical variables are checked for:

- Missing values
- Inconsistent capitalization
- Leading/trailing whitespace
- Unexpected category labels
- Duplicate representations of the same category

The main categorical variables include:

- `productcategory`
- `paymentmethod`
- `customerlocation`
- `customersegment`

Standardizing these values ensures that equivalent categories are not
treated as separate groups during SQL analysis, EDA, or Power BI reporting.

In [16]:
categorical_columns = df.select_dtypes(include='object')

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.title()
    )

In [17]:
category_mapping = {
    "Electrnics": "Electronics",
    "Fasion": "Fashion"
}

df["productcategory"] = (
    df["productcategory"]
    .replace(category_mapping)
)

In [18]:
df["productcategory"].value_counts()

productcategory
Electronics       197
Books             172
Home & Kitchen    172
Beauty            162
Toys              153
Fashion           144
Name: count, dtype: Int64

In [19]:
for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


orderid
orderid
Ord100000    1
Ord100001    1
Ord100002    1
Ord100003    1
Ord100004    1
            ..
Ord100995    1
Ord100996    1
Ord100997    1
Ord100998    1
Ord100999    1
Name: count, Length: 1000, dtype: Int64

productcategory
productcategory
Electronics       197
Books             172
Home & Kitchen    172
Beauty            162
Toys              153
Fashion           144
Name: count, dtype: Int64

paymentmethod
paymentmethod
Net Banking         270
Cash On Delivery    250
Credit Card         241
Upi                 239
Name: count, dtype: Int64

customerlocation
customerlocation
Chennai      174
Delhi        173
Bangalore    165
Mumbai       165
Pune         165
Hyderabad    158
Name: count, dtype: Int64

customersegment
customersegment
New        355
Regular    353
Vip        292
Name: count, dtype: Int64


In [20]:
quantity_median_by_category = (
    df.groupby("productcategory")["quantity"]
      .transform("median")
)

df["quantity"] = df["quantity"].fillna(
    quantity_median_by_category
)

df["quantity"] = df["quantity"].fillna(
    df["quantity"].median()
)

In [21]:
df["quantity"] = df["quantity"].round().astype(int)

In [22]:
price_median_by_category = (
    df.groupby("productcategory")["priceperunit"]
      .transform("median")
)

df["priceperunit"] = df["priceperunit"].fillna(
    price_median_by_category
)

df["priceperunit"] = df["priceperunit"].fillna(
    df["priceperunit"].median()
)

## Numerical Data Validation

Numerical variables are examined for values that are logically impossible
or inconsistent with the business context.

The following variables require particular attention:

- `quantity`
- `priceperunit`
- `discountapplied`
- `deliverytimedays`
- `totalamount`

Examples of potential data-quality issues include:

- Negative quantities
- Negative prices
- Discounts outside the expected range
- Negative delivery times
- Invalid monetary amounts

Each issue is investigated before a cleaning rule is applied.

In [24]:
print("Invalid quantity:",
      (df["quantity"] <= 0).sum())

print("Invalid price:",
      (df["priceperunit"] <= 0).sum())

print("Invalid discount:",
      ((df["discountapplied"] < 0) |
       (df["discountapplied"] > 1)).sum())

print("Invalid delivery time:",
      (df["deliverytimedays"] < 0).sum())

print("Invalid target values:",
      (~df["isreturned"].isin([0, 1])).sum())

Invalid quantity: 0
Invalid price: 0
Invalid discount: 0
Invalid delivery time: 50
Invalid target values: 0


In [34]:
df.loc[
    df["deliverytimedays"] < 0,
    "deliverytimedays"
] = np.nan

In [36]:
delivery_median_by_category = (
    df.groupby("productcategory")["deliverytimedays"]
      .transform("median")
)

df["deliverytimedays"] = df["deliverytimedays"].fillna(
    delivery_median_by_category
)

In [38]:
df["deliverytimedays"] = df["deliverytimedays"].fillna(
    df["deliverytimedays"].median()
)

In [39]:
print(
    "Invalid delivery time:",
    (df["deliverytimedays"] < 0).sum()
)

print(
    "Missing delivery time:",
    df["deliverytimedays"].isna().sum()
)

Invalid delivery time: 0
Missing delivery time: 0


In [25]:
df["discountapplied"].describe()

count    1000.000000
mean        0.147130
std         0.086017
min         0.000000
25%         0.070000
50%         0.145000
75%         0.220000
max         0.300000
Name: discountapplied, dtype: float64

In [26]:
assert df["discountapplied"].between(0, 1).all()

## Business Logic Validation

In addition to checking individual columns, relationships between variables
are examined.

Examples include:

- Order value should be consistent with transaction-level information.
- Return status should contain only valid binary values.
- Discount values should be consistent with the defined business rules.
- Order identifiers should uniquely represent orders.

These checks help identify records that may be technically valid from a
data-type perspective but logically inconsistent from a business
perspective.

In [27]:
df["calculated_totalamount"] = (
    df["quantity"]
    * df["priceperunit"]
    * (1 - df["discountapplied"])
)

In [28]:
df["calculated_totalamount"] = (
    df["calculated_totalamount"]
    .round(2)
)
df["totalamount"] = df["calculated_totalamount"]

In [29]:
df = df.drop(
    columns=["calculated_totalamount"]
)

### Return Target Validation

The target variable `isreturned` is critical because it will be used as the
target variable for the machine-learning classification problem.

It is therefore validated to ensure that:

- Only valid binary values are present
- No unexpected labels exist
- Missing target values are handled appropriately

Expected values:

```text
0 → Order was not returned
1 → Order was returned

In [30]:
print(df["isreturned"].value_counts())

print(
    df["isreturned"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

isreturned
0    646
1    354
Name: count, dtype: int64
isreturned
0    64.6
1    35.4
Name: proportion, dtype: float64


In [31]:
df["order_month"] = df["orderdate"].dt.month

In [32]:
df.head()

,orderid,orderdate,productcategory,quantity,priceperunit,paymentmethod,customerlocation,customersegment,discountapplied,deliverytimedays,isreturned,totalamount,order_month
0,Ord100000,2023-06-26,Electronics,4,3968.220,Credit Card,Bangalore,New,0.18,4,0,13015.76,6
1,Ord100001,2023-09-24,Books,3,1554.250,Net Banking,Chennai,New,0.20,5,1,3730.20,9
2,Ord100002,2023-12-07,Books,4,2752.710,Cash On Delivery,Delhi,Vip,0.05,-5,1,10460.30,12
3,Ord100003,2023-10-09,Books,1,2765.735,Credit Card,Chennai,New,0.29,1,0,1963.67,10
4,Ord100004,2023-03-04,Books,2,2765.735,Net Banking,Chennai,Regular,0.16,7,0,4646.43,3


## Final Data Validation

Before exporting the dataset, a final validation pass is performed.

The cleaned dataset must satisfy the following conditions:

- No unexpected duplicate records
- No invalid delivery-time values
- No invalid prices or quantities
- Valid categorical values
- Valid return labels
- Correct data types
- Required fields contain no unresolved missing values
- Derived features are correctly generated

Only after these checks pass is the dataset considered ready for downstream
SQL ingestion and analysis.

In [42]:
validation_checks = {
    "No duplicate OrderIDs": (
        df["orderid"].duplicated().sum() == 0
    ),

    "No missing values": (
        df.isnull().sum().sum() == 0
    ),

    "Quantity > 0": (
        df["quantity"] > 0
    ).all(),

    "Price > 0": (
        df["priceperunit"] > 0
    ).all(),

    "Discount between 0 and 1": (
        df["discountapplied"].between(0, 1)
    ).all(),

    "Delivery time >= 0": (
        df["deliverytimedays"] >= 0
    ).all(),

    "Valid target values": (
        df["isreturned"].isin([0, 1])
    ).all(),

    "Total amount >= 0": (
        df["totalamount"] >= 0
    ).all()
}

validation_results = pd.DataFrame(
    validation_checks.items(),
    columns=["Validation_Check", "Passed"]
)

validation_results

,Validation_Check,Passed
0,No duplicate OrderIDs,True
1,No missing values,True
2,Quantity > 0,True
3,Price > 0,True
4,Discount between 0 and 1,True
5,Delivery time >= 0,True
6,Valid target values,True
7,Total amount >= 0,True


### Validation Result

The final validation confirms that the cleaned dataset satisfies the
defined data-quality rules.

The resulting dataset is now suitable for:

1. Loading into MySQL
2. SQL-based business analysis
3. Exploratory data analysis
4. Power BI visualization
5. Machine-learning preprocessing and modeling

In [43]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

validation_results.to_csv(
    OUTPUT_PATH,
    index=False
)

In [44]:
CLEANED_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    CLEANED_PATH,
    index=False
)

print(f"Cleaned dataset saved to: {CLEANED_PATH}")

Cleaned dataset saved to: ..\data\cleaned\ecommerce_cleaned.csv


# 15. Summary

The data-cleaning and validation stage has prepared the e-commerce
transaction dataset for downstream analysis.

### Completed

- Loaded and profiled the raw dataset
- Inspected data types and structure
- Identified duplicate records
- Investigated missing values
- Validated numerical variables
- Removed/handled invalid delivery-time records
- Validated categorical variables
- Validated the return target
- Created the `order_month` feature
- Performed final quality checks
- Exported the cleaned dataset

### Data Flow

```text
Raw E-Commerce Data
        ↓
Data Profiling
        ↓
Data Cleaning
        ↓
Business Rule Validation
        ↓
Final Validation
        ↓
ecommerce_cleaned.csv
        ↓
MySQL / EDA / Power BI / ML